# The risk of primary cesarean delivery among multiparous women with no history of cesarean section

## Setup

In [ ]:
import sweetviz as sv
import config as c
import eda_utils as eu
from IPython.display import display
import installations as install
import importlib
import pandas as pd
import numpy as np
importlib.reload(install)
importlib.reload(c)
importlib.reload(eu)

# I reemind you to check the path way in the config file

In [ ]:
# Checking the installed libraries
install.check_installed_packages()

## Cohort Definition

In [ ]:
pre_df = c.HOLY_DATA.copy()


In [ ]:
pre_df.shape

In [ ]:
pre_df.info()

In [ ]:

# ------- Not Relevant ---------------------------------------------------------------
# A: Only term pregnancies - 37 weeks and above (according to the Robinson Group 3, 4)
# df_filtered = pre_df[pre_df['gestational_age_weeks'] >= 37].copy()

# B: Only multiparous women
# df_filtered = df_filtered[df_filtered['parity'] >= 1]

# ------- Relevant to the model building section (Notebook 2) ------------------------
# C: Only women without a planned cesarean section
# df_filtered = pre_df[pre_df['was_planned_cs'] == 0]
# df_filtered = df_filtered.drop(columns=['was_planned_cs'])

# df_ready_for_eda = df_filtered


df_ready_for_eda = pre_df



df_ready_for_eda.shape


## EDA - Exploratory Data Analysis

### Part 1: Univariate Analysis - Central Tendency And Dispersion

In [ ]:
report = sv.analyze(df_ready_for_eda)
report.show_html(c.OUTPUT_DIR + "/Sweetviz_EDA_Automatic_Report.html")

In [ ]:
duplicates = df_ready_for_eda.duplicated(keep=False)
duplicate_count = duplicates.sum()
print(f"Number of identical duplicate rows: {duplicate_count}")

In [ ]:
eu.visualize_feature_distributions(df_ready_for_eda)

In [ ]:
# Setting up the data frame after initial diagnosis,
# update c.gdm_schema if necessary before running

df = eu.apply_data_schema(df_ready_for_eda, c.gdm_schema)
parity_order = [1, 2, 3, 4, 5, 6, 7, 8]

df['parity'] = pd.Categorical(df['parity'],
                              categories=parity_order, ordered=True)

In [ ]:
eu.describe_numerical(df)

In [ ]:
eu.describe_categorical(df)

In [ ]:
eu.visualize_feature_distributions(df)

In [ ]:
(outliers_matrix, count_of_outliers,
 rare_categories_cat_list) = eu.visualize_outliers_and_proportions(df)

In [ ]:
print(count_of_outliers)

In [ ]:
print(*rare_categories_cat_list, sep="\n")

In [ ]:
# You must decide whether there are normal variables and if so,
# which ones they are and update under the variable - normal_cont_vars
normal_cont_vars = ['height_cm', 'hemoglobin_first', 'birth_weight_g']

In [ ]:
(cat_vars,
 bin_cat,
 multy_cat,
 non_normal_cont_vars,
 normal_cont_vars,
 datetime_vars) = eu.split_variables_by_type(df, normal_cont_vars)

print(cat_vars, bin_cat, multy_cat, non_normal_cont_vars,
      normal_cont_vars, datetime_vars, sep="\n\n")

### Part 2: Bivariate Analysis and Statistical Correlations

#### Pearson test

In [ ]:
(pearson_pval_matrix,
 pearson_corr_matrix) = eu.pearson_matrices(
     df=df,
     normal_vars=normal_cont_vars
 )

In [ ]:
eu.plot_statistical_heatmaps(
    pval_matrix=pearson_pval_matrix,
    effect_matrix=pearson_corr_matrix,
    plot_title="Pearson Correlation",
    test_name="Pearson r",
    file_name="1_pearson_heatmap")

In [ ]:
res_pearson_sig = eu.significance_table(pval_matrix=pearson_pval_matrix,
                                        stat_matrix=pearson_corr_matrix,
                                        test_name="pearson", alpha=c.ALPHA_LEVEL,
                                        effect_threshold=c.GLOBAL_STAT_THRESHOLDS["pearson"])


eu.save_df(df=res_pearson_sig, file_name="pearson_summary")

# display(res_pearson_sig)

display((res_pearson_sig[res_pearson_sig["Is_Significant"] & res_pearson_sig["Is_Meaningful_Effect"]]).shape)

#### Spearman test

In [ ]:
spearman_pval_matrix, spearman_corr_matrix = eu.spearman_matrices(
    df=df,
    normal_vars=normal_cont_vars,
    non_normal_vars=non_normal_cont_vars
)

In [ ]:
eu.plot_statistical_heatmaps(
    pval_matrix=spearman_pval_matrix,
    effect_matrix=spearman_corr_matrix,
    plot_title="Spearman Correlation",
    test_name="Spearman Rank r",
    file_name="2_spearman_heatmap")

In [ ]:
res_spearman_sig = eu.significance_table(pval_matrix=spearman_pval_matrix,
                                         stat_matrix=spearman_corr_matrix,
                                         test_name="spearman", alpha=c.ALPHA_LEVEL,
                                         effect_threshold=c.GLOBAL_STAT_THRESHOLDS["spearman"])

eu.save_df(df=res_spearman_sig, file_name="spearman_summary")

# display(res_spearman_sig)

display((res_spearman_sig[res_spearman_sig["Is_Significant"] & res_spearman_sig["Is_Meaningful_Effect"]]).shape)

#### Mann-Whitney

In [ ]:
mann_whitney_pval_matrix, mann_whitney_effect_matrix = eu.mann_whitney_matrices(df=df, non_normal_vars=non_normal_cont_vars, binary_vars=bin_cat)

In [ ]:
eu.plot_statistical_heatmaps(
    pval_matrix=mann_whitney_pval_matrix,
    effect_matrix=mann_whitney_effect_matrix,
    plot_title="Mann-Whitney U (Rank-Biserial r)",
    test_name="Mann-Whitney U (Rank-Biserial r)",
    file_name="3_mann_whitney_heatmap"
)

In [ ]:
res_mannwhitney_sig = eu.significance_table(pval_matrix=mann_whitney_pval_matrix,
                                            stat_matrix=mann_whitney_effect_matrix,
                                            test_name="mannwhitney", alpha=c.ALPHA_LEVEL,
                                            effect_threshold=c.GLOBAL_STAT_THRESHOLDS["mannwhitney"])

eu.save_df(df=res_mannwhitney_sig, file_name="mannwhitney_summary")

# display(res_mannwhitney_sig)

display((res_mannwhitney_sig[res_mannwhitney_sig["Is_Significant"] & res_mannwhitney_sig["Is_Meaningful_Effect"]]).shape)

#### t-test

In [ ]:
# please check this line 
ttest_pval_matrix, ttest_effect_matrix = eu.ttest_matrices(df=df, normal_vars=normal_cont_vars, binary_vars=bin_cat)

In [ ]:
eu.plot_statistical_heatmaps(
    pval_matrix=ttest_pval_matrix,
    effect_matrix=ttest_effect_matrix,
    plot_title="T-Test (Point-Biserial r)",
    test_name="T-Test (Point-Biserial r)",
    file_name="4_ttest_heatmap"
)

In [ ]:
res_ttest_sig = eu.significance_table(pval_matrix=ttest_pval_matrix,
                                      stat_matrix=ttest_effect_matrix,
                                      test_name="ttest", alpha=c.ALPHA_LEVEL,
                                      effect_threshold=c.GLOBAL_STAT_THRESHOLDS["ttest"])

eu.save_df(df=res_ttest_sig, file_name="ttest_summary")

# display(res_ttest_sig)

display((res_ttest_sig[res_ttest_sig["Is_Significant"] & res_ttest_sig["Is_Meaningful_Effect"]]).shape)

#### Chi test

In [ ]:
chi2_pval_matrix, chi2_cramers_v_matrix = eu.chi2_matrices(df=df, cat_vars=cat_vars)

In [ ]:
eu.plot_statistical_heatmaps(
    pval_matrix=chi2_pval_matrix,
    effect_matrix=chi2_cramers_v_matrix,
    plot_title="Chi-Square",
    test_name="Chi-Square (Cramér's V)",
    file_name="5_chi2_heatmap"
)

In [ ]:
res_chi2_sig = eu.significance_table(pval_matrix=chi2_pval_matrix,
                                     stat_matrix=chi2_cramers_v_matrix,
                                     test_name="chi2", alpha=c.ALPHA_LEVEL,
                                     effect_threshold=c.GLOBAL_STAT_THRESHOLDS["chi2"])

eu.save_df(df=res_chi2_sig, file_name="chi2_summary")

# display(res_chi2_sig)

display((res_chi2_sig[res_chi2_sig["Is_Significant"] & res_chi2_sig["Is_Meaningful_Effect"]]).shape)

#### Anova

In [ ]:
anova_pval_matrix, anova_effect_matrix = eu.anova_matrices(
    df=df, normal_vars=normal_cont_vars, cat_vars_gt2=multy_cat)

In [ ]:
eu.plot_statistical_heatmaps(
    pval_matrix=anova_pval_matrix,
    effect_matrix=anova_effect_matrix,
    plot_title="ANOVA (Eta-Squared)",
    test_name="ANOVA (Eta-Squared)",
    file_name="6_anova_heatmap")

In [ ]:
res_anova_sig = eu.significance_table(pval_matrix=anova_pval_matrix,
                                      stat_matrix=anova_effect_matrix,
                                      test_name="anova", alpha=c.ALPHA_LEVEL,
                                      effect_threshold=c.GLOBAL_STAT_THRESHOLDS["anova"])

eu.save_df(df=res_anova_sig, file_name="anova_summary")

# display(res_anova_sig)

display((res_anova_sig[res_anova_sig["Is_Significant"] & res_anova_sig["Is_Meaningful_Effect"]]).shape)

#### Kruskal H-Test

In [ ]:
kruskal_pval_matrix, kruskal_effect_matrix = eu.htest_matrices(
    df=df, non_normal_vars=non_normal_cont_vars, cat_vars_gt2=multy_cat)

In [ ]:
eu.plot_statistical_heatmaps(
    pval_matrix=kruskal_pval_matrix,
    effect_matrix=kruskal_effect_matrix,
    plot_title="Kruskal-Wallis (Epsilon-Squared)",
    test_name="Kruskal-Wallis (Epsilon-Squared)",
    file_name="7_kruskal_heatmap")

In [ ]:
res_htest_sig = eu.significance_table(pval_matrix=kruskal_pval_matrix,
                                      stat_matrix=kruskal_effect_matrix,
                                      test_name="htest", alpha=c.ALPHA_LEVEL,
                                      effect_threshold=c.GLOBAL_STAT_THRESHOLDS["htest"])

eu.save_df(df=res_htest_sig, file_name="htest_summary")

# display(res_htest_sig)

display((res_htest_sig[res_htest_sig["Is_Significant"] & res_htest_sig["Is_Meaningful_Effect"]]).shape)

#### Summary

In [ ]:
cor_summary_sig = pd.concat([res_pearson_sig, res_spearman_sig, res_mannwhitney_sig, res_ttest_sig, res_chi2_sig, res_anova_sig, res_htest_sig], ignore_index=True)

cssig = cor_summary_sig[cor_summary_sig["Is_Significant"] & cor_summary_sig["Is_Meaningful_Effect"]]

cssig.reset_index(drop=True, inplace=True)

eu.save_df(df=cssig, file_name="statistical_summary")



### Part 3: Outlier Detection And Clinical Logic Anomalies

In [ ]:
print(count_of_outliers)

In [ ]:
print(*rare_categories_cat_list, sep="\n")

In [ ]:
eu.plot_outliers_heatmap(df=df, outliers_matrix=outliers_matrix, show=True)

In [ ]:
# The following function is a beta function only
# and will not be run in sequence until after we have
# tested everything up to this point.

df_clean_raw, report_df = eu.apply_clinical_logic(df)

eu.save_df(report_df,"outlier_detection_report")

display(report_df)

#### Categories smaller than 5%

In [ ]:
rare_categories_report_df = eu.categorical_frequencies_table(
    df=df_clean_raw,
    cat_vars=cat_vars,
    threshold=0.05
)



eu.save_df(rare_categories_report_df, file_name="categories_report")
display(rare_categories_report_df)

In [ ]:
display(rare_categories_report_df[rare_categories_report_df["Is_Rare"]].reset_index(drop=True))

#####  start_mode_raw

In [ ]:
# start_mode_raw

s1_raw = df_clean_raw['start_mode_raw'].copy()

SPONTANEOUS_CODES = [1, 6, 102, 104, 107]
INDUCED_CODES = [3, 4, 5, 105, 106, 108, 109, 110, 111, 112, 113]
PLANNED_CS_CODES = [2, 101]    # kept in cohort, own category (these codes were used to derive was_planned_cs)
NULL_ONSET_CODES = [103]  # documented null onset -> NaN   


CAT_SPONTANEOUS = 1
CAT_INDUCED = 2
CAT_PLANNED_CS = 3


mapping_dict = {}
for code in SPONTANEOUS_CODES: mapping_dict[code] = CAT_SPONTANEOUS
for code in INDUCED_CODES:     mapping_dict[code] = CAT_INDUCED
for code in PLANNED_CS_CODES:  mapping_dict[code] = CAT_PLANNED_CS
for code in NULL_ONSET_CODES:  mapping_dict[code] = np.nan

start_mode_raw_mapped = s1_raw.map(mapping_dict)
frequencies_1 = start_mode_raw_mapped.value_counts(dropna=False)

summary_dict = {
    CAT_SPONTANEOUS: (SPONTANEOUS_CODES, 'Spontaneous', frequencies_1.get(CAT_SPONTANEOUS, 0)),
    CAT_INDUCED:     (INDUCED_CODES, 'Induced', frequencies_1.get(CAT_INDUCED, 0)),
    CAT_PLANNED_CS:  (PLANNED_CS_CODES, 'Planned CS', frequencies_1.get(CAT_PLANNED_CS, 0)),
    'NaN_Values':    (NULL_ONSET_CODES, 'Missing', frequencies_1.get(np.nan, 0))
}

print(" Start Mode Dictionary Summary")
for new_cat, details in summary_dict.items():
    print(f"Category {new_cat}: Codes {details[0]} | Label: '{details[1]}' | Frequency: {details[2]}")

print("-"*100)
print("\n Vector Value Counts (start_mode_raw_mapped)")
print(start_mode_raw_mapped.value_counts(dropna=False))

##### Membranes_color_raw

In [ ]:
import pandas as pd
import numpy as np

# membranes_color_raw
s2_raw = df_clean_raw['membranes_color_raw'].copy()

CLEAN_CODES = [1, 2]                 # Clean amniotic fluid
MECONIUM_CODES = [4, 5, 6, 101, 102, 103]   # Meconium-stained amniotic fluid
BLOODY_CODES = [3]                   # Bloody amniotic fluid

# Splitting the NULL group
NULL_NO_OBS_CODES = [104, 105, 108] # Not observed / Intact membranes
NULL_QUANTITY_CODES = [106, 107] # Quantitative information (majority/minority) mistakenly entered into the color column

CAT_CLEAN = 1
CAT_MECONIUM = 2
CAT_BLOODY = 3

color_mapping_dict = {}
for code in CLEAN_CODES: color_mapping_dict[code] = CAT_CLEAN
for code in MECONIUM_CODES: color_mapping_dict[code] = CAT_MECONIUM
for code in BLOODY_CODES: color_mapping_dict[code] = CAT_BLOODY
for code in NULL_NO_OBS_CODES: color_mapping_dict[code] = np.nan
for code in NULL_QUANTITY_CODES: color_mapping_dict[code] = np.nan 

membranes_color_raw_mapped = s2_raw.map(color_mapping_dict)
color_frequencies = membranes_color_raw_mapped.value_counts(dropna=False)

summary_color_dict = {
    CAT_CLEAN:    (CLEAN_CODES, 'Clean', color_frequencies.get(CAT_CLEAN, 0)),
    CAT_MECONIUM: (MECONIUM_CODES, 'Meconium', color_frequencies.get(CAT_MECONIUM, 0)),
    CAT_BLOODY:   (BLOODY_CODES, 'Bloody', color_frequencies.get(CAT_BLOODY, 0)),
    'NaN_No_Obs': (NULL_NO_OBS_CODES, 'No observation / Intact', s2_raw.isin(NULL_NO_OBS_CODES).sum()),
    'NaN_Quantity':(NULL_QUANTITY_CODES, 'Quantity Info (Mapped to NaN)', s2_raw.isin(NULL_QUANTITY_CODES).sum())
}

print(" Membranes Color Dictionary Summary")
for new_cat, details in summary_color_dict.items():
    print(f"Category {new_cat}: Codes {details[0]} | Label: '{details[1]}' | Frequency: {details[2]}")

print("-" * 100)
print("\n Vector Value Counts (membranes_color_raw_mapped)")
print(membranes_color_raw_mapped.value_counts(dropna=False))

##### Parity

In [ ]:
# parity
s3_raw = df_clean_raw['parity'].copy()

P1_CODES = [1]
P2_CODES = [2]
P3_CODES = [3]
P4_CODES = [4]
P5_CODES = [5]
P6_PLUS_CODES = [6, 7, 8] 

CAT_P1 = 1
CAT_P2 = 2
CAT_P3 = 3
CAT_P4 = 4
CAT_P5 = 5
CAT_P6_PLUS = 6

parity_mapping_dict = {}
for code in P1_CODES:      parity_mapping_dict[code] = CAT_P1
for code in P2_CODES:      parity_mapping_dict[code] = CAT_P2
for code in P3_CODES:      parity_mapping_dict[code] = CAT_P3
for code in P4_CODES:      parity_mapping_dict[code] = CAT_P4
for code in P5_CODES:      parity_mapping_dict[code] = CAT_P5
for code in P6_PLUS_CODES: parity_mapping_dict[code] = CAT_P6_PLUS

parity_mapped = s3_raw.map(parity_mapping_dict)

parity_frequencies = parity_mapped.value_counts(dropna=False)

summary_parity_dict = {
    CAT_P1:      (P1_CODES, 'Parity 1', parity_frequencies.get(CAT_P1, 0)),
    CAT_P2:      (P2_CODES, 'Parity 2', parity_frequencies.get(CAT_P2, 0)),
    CAT_P3:      (P3_CODES, 'Parity 3', parity_frequencies.get(CAT_P3, 0)),
    CAT_P4:      (P4_CODES, 'Parity 4', parity_frequencies.get(CAT_P4, 0)),
    CAT_P5:      (P5_CODES, 'Parity 5', parity_frequencies.get(CAT_P5, 0)),
    CAT_P6_PLUS: (P6_PLUS_CODES, 'Parity 6+', parity_frequencies.get(CAT_P6_PLUS, 0)),
    'NaN_Values': (['Null'], 'Missing', parity_frequencies.get(np.nan, 0))
}

print(" Parity Dictionary Summary")
for new_cat, details in summary_parity_dict.items():
    codes_display = str(details[0]) if len(details[0]) < 10 else f"{str(details[0][:5])[:-1]}, ...]"
    print(f"Category {new_cat}: Codes {codes_display:<15} | Label: '{details[1]:<10}' | Frequency: {details[2]}")

print("-"*100)
print("\n Vector Value Counts (parity_mapped)")
print(parity_mapped.value_counts(dropna=False).sort_index())

##### Membranes_type_raw

In [ ]:
# membranes_type_raw
s4_raw = df_clean_raw['membranes_type_raw'].copy()

SPONTANEOUS_TYPE_CODES = [1, 5, 7, 101, 111]
ARTIFICIAL_TYPE_CODES  = [4, 8, 102]

NULL_PROM_CODES     = [2, 3, 6]                 # Premature Rupture of Membranes
NULL_INTACT_CODES   = [9, 103, 108, 110, 113]   # Intact / Not ruptured
NULL_OTHER_CODES    = [10, 17, 109, 112]        # Unknown time / Procedure / Missing from dictionary

CAT_SPONTANEOUS_TYPE = 0
CAT_ARTIFICIAL_TYPE  = 1

type_mapping_dict = {}
for code in SPONTANEOUS_TYPE_CODES:
    type_mapping_dict[code] = CAT_SPONTANEOUS_TYPE
for code in ARTIFICIAL_TYPE_CODES:
    type_mapping_dict[code] = CAT_ARTIFICIAL_TYPE
for code in NULL_PROM_CODES:
    type_mapping_dict[code] = np.nan
for code in NULL_INTACT_CODES:
    type_mapping_dict[code] = np.nan
for code in NULL_OTHER_CODES:
    type_mapping_dict[code] = np.nan

membranes_type_raw_mapped = s4_raw.map(type_mapping_dict)

type_frequencies = membranes_type_raw_mapped.value_counts(dropna=False)

summary_type_dict = {
    CAT_SPONTANEOUS_TYPE: (SPONTANEOUS_TYPE_CODES, 'Spontaneous (incl. post-exam)', type_frequencies.get(CAT_SPONTANEOUS_TYPE, 0)),
    CAT_ARTIFICIAL_TYPE:  (ARTIFICIAL_TYPE_CODES,  'Artificial (AROM)', type_frequencies.get(CAT_ARTIFICIAL_TYPE, 0)),
    'NaN_PROM':           (NULL_PROM_CODES,        'Mapped to NaN: PROM/PPROM', s4_raw.isin(NULL_PROM_CODES).sum()),
    'NaN_Intact':         (NULL_INTACT_CODES,      'Mapped to NaN: Intact/Unruptured', s4_raw.isin(NULL_INTACT_CODES).sum()),
    'NaN_Other':          (NULL_OTHER_CODES,       'Mapped to NaN: Unknown/Procedure', s4_raw.isin(NULL_OTHER_CODES).sum()),
    'NaN_Unmapped':       (['NaN'],                     'Originally Missing or Unmapped', s4_raw.isna().sum())
}

print(" Membranes Type Dictionary Summary")
for new_cat, details in summary_type_dict.items():
    print(f"Category {new_cat:<20}: Codes {str(details[0]):<20} | Label: '{details[1]:<35}' | Frequency: {details[2]}")

print("-" * 100)
print("\n Vector Value Counts (membranes_type_raw_mapped)")
print(membranes_type_raw_mapped.value_counts(dropna=False))

##### Summary

In [ ]:
# end of cleaning Outlier

df_clean = df_clean_raw.copy()
df_clean['start_mode(clean)'] = start_mode_raw_mapped.astype('category')
df_clean['membranes_color(clean)'] = membranes_color_raw_mapped.astype('category')
df_clean['parity(clean)'] = pd.Categorical(parity_mapped, ordered=True)
df_clean['membranes_type(clean)'] = membranes_type_raw_mapped.astype('category')
new_bin_cols = ['membranes_type(clean)']
new_multy_cols = ['start_mode(clean)', 'membranes_color(clean)', 'parity(clean)']
df_clean.shape


### Part 4: Missing Data Analysis

In [ ]:
eu.plot_simple_missing_heatmap(
    df=df_clean,
    rand=False,
    show=True,
    save_plt=True, end=(" [post]"))

In [ ]:
missingness_report_df = eu.missingness_mechanism_table(
    df=df_clean,
    normal_vars=normal_cont_vars,
    non_normal_vars=non_normal_cont_vars,
    multy_cat_vars=multy_cat + new_multy_cols,
    bin_cat_vars=bin_cat + new_bin_cols
)

eu.save_df(missingness_report_df,"missingness_report")
display(missingness_report_df)

In [ ]:
# Data Preparation: Feature Selection
# Divide the features into the following three groups and update 'config.py':
# 1. MAIN_MODEL_VARS: Primary predictive variables
# 2. SENSITIVITY_VARS: Variables for sensitivity analysis
# 3. LIMITATION_VARS: Variables identifying study limitations

### Part 5 : Feature Engineering 

In [ ]:
final_df = eu.set_log(df=df_clean, numeric_cols=[
    var for var in non_normal_cont_vars if var != 'weight_gain'])

new_log_cols = [col for col in final_df.columns if col.startswith("log(")]
print(new_log_cols)
display(eu.describe_numerical(final_df[new_log_cols]))
eu.visualize_feature_distributions(df=final_df[new_log_cols + [c.TARGET_VAR]], end_file_name="_FE_df")


### Part 6: Table One

#### Main Table one

In [ ]:
table1 = eu.table1(df=final_df,normal_vars=normal_cont_vars, non_normal_vars= non_normal_cont_vars + new_log_cols,
                   multy_cat_vars= multy_cat + new_multy_cols,
                   bin_cat_vars=bin_cat + new_bin_cols)

table1.reset_index(drop=True, inplace=True)

eu.save_df(df=table1, file_name="TableOne")

display(table1.head(-10), table1.shape)

#### Secondary Table one (bmi_computed)

In [ ]:
table1_s_bmi_computed = eu.sensitivity_table1(df=final_df,normal_vars=normal_cont_vars,
                               non_normal_vars= non_normal_cont_vars + new_log_cols,
                               multy_cat_vars= multy_cat + new_multy_cols,
                               bin_cat_vars=bin_cat + new_bin_cols,
                               target_col="bmi_computed",target_name="BMI Computed")



eu.save_df(df=table1_s_bmi_computed, file_name="TableOneSensitivity_bmi_computed")

display(table1_s_bmi_computed.head(5), table1_s_bmi_computed.shape)

#### Thirdly Table one (weight_gain)

In [ ]:
table1_s_weight_gain = eu.sensitivity_table1(df=final_df,normal_vars=normal_cont_vars,
                               non_normal_vars= non_normal_cont_vars + new_log_cols,
                               multy_cat_vars= multy_cat + new_multy_cols,
                               bin_cat_vars=bin_cat + new_bin_cols,
                               target_col="weight_gain",target_name="Weight Gain")



eu.save_df(df=table1_s_weight_gain, file_name="TableOneSensitivity_weight_gain")

display(table1_s_weight_gain.head(5), table1_s_weight_gain.shape)

#### 4th Table one (weight_at_admission)

In [ ]:
table1_s_weight_at_admission = eu.sensitivity_table1(df=final_df,normal_vars=normal_cont_vars,
                               non_normal_vars= non_normal_cont_vars + new_log_cols,
                               multy_cat_vars= multy_cat + new_multy_cols,
                               bin_cat_vars=bin_cat + new_bin_cols,
                               target_col="weight_at_admission",target_name="Weight At Admission")


eu.save_df(df=table1_s_weight_at_admission, file_name="TableOneSensitivity_weight_at_admission")

display(table1_s_weight_at_admission.head(5), table1_s_weight_at_admission.shape)


#### 5th Table one (height_cm)

In [ ]:
table1_s_height_cm = eu.sensitivity_table1(df=final_df,normal_vars=normal_cont_vars,
                               non_normal_vars= non_normal_cont_vars + new_log_cols,
                               multy_cat_vars= multy_cat + new_multy_cols,
                               bin_cat_vars=bin_cat + new_bin_cols,
                               target_col="height_cm",target_name="Height CM")


eu.save_df(df=table1_s_height_cm, file_name="TableOneSensitivity_height_cm")

display(table1_s_height_cm.head(5), table1_s_height_cm.shape)

#### 6th Table one (weight_pre_pregnancy)

In [ ]:
table1_s_weight_pre_pregnancy = eu.sensitivity_table1(df=final_df,normal_vars=normal_cont_vars,
                               non_normal_vars= non_normal_cont_vars + new_log_cols,
                               multy_cat_vars= multy_cat + new_multy_cols,
                               bin_cat_vars=bin_cat + new_bin_cols,
                               target_col="weight_pre_pregnancy",target_name="Weight_Pre_Pregnancy")


eu.save_df(df=table1_s_weight_pre_pregnancy, file_name="TableOneSensitivity_weight_pre_pregnancy")

display(table1_s_weight_pre_pregnancy.head(5), table1_s_weight_pre_pregnancy.shape)

## Save Dataframe

In [ ]:
eu.save_df(df=final_df, file_name="df_for_model", path=c.DATA_DIR)
print(final_df.columns)
final_df.head(5)